# Chicken Threat Detector Model v3

This notebook is the **training and evaluation stage**. It intentionally starts from a ZIP exported by `ChickenThreadDetectorDatav3.ipynb`.

Pipeline boundary:

- Upload or point to `chicken_threat_dataset_v3.zip`.
- Unpack the YOLO dataset.
- Validate `data.yaml`, labels, and split structure.
- Train a YOLO detector with stronger augmentation and training optimizers.
- Evaluate on `valid` and `test`.
- Export best weights and run a small inference sanity check.

Default model: `yolo11s.pt`, because it matches the v2 baseline and is a practical Colab/edge-deployment compromise. Use `yolo11n.pt` for faster experiments or `yolo11m.pt` for higher accuracy if GPU budget allows.


## Install dependencies

Run once per runtime. Restart the runtime if Colab asks for it, then continue from the next cell.


In [ ]:
%pip install -q --upgrade ultralytics pyyaml matplotlib pandas tqdm opencv-python-headless "pillow>=10.4.0,<11.0.0"

## Runtime setup

In [ ]:
import os
import random
import shutil
import sys
import zipfile
from collections import Counter
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from PIL import Image
from tqdm.auto import tqdm

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in str(get_ipython())
USE_GOOGLE_DRIVE = False

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("CHICKEN_THREAT_WORKDIR", "/content/drive/MyDrive/chicken_threat_detector_v3_model")
elif IN_COLAB:
    os.environ.setdefault("CHICKEN_THREAT_WORKDIR", "/content/chicken_threat_detector_v3_model")

WORK_DIR = Path(os.getenv("CHICKEN_THREAT_WORKDIR", "data/chicken_threat_detector_v3_model")).resolve()
IMPORT_ROOT = WORK_DIR / "imported_dataset"
RUNS_ROOT = WORK_DIR / "runs"
EXPORT_ROOT = WORK_DIR / "exports"
for directory in (WORK_DIR, IMPORT_ROOT, RUNS_ROOT, EXPORT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

SPLITS = ("train", "valid", "test")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CANONICAL_NAMES = [
    "chicken",
    "other_poultry",
    "rodent",
    "fox",
    "cat",
    "dog",
    "marten_weasel",
    "bird_of_prey",
    "other_bird",
    "person",
]

print(f"IN_COLAB={IN_COLAB}")
print(f"WORK_DIR={WORK_DIR}")


## Upload and unpack dataset ZIP

Use the ZIP exported by the data notebook. In Colab, the cell opens a file upload dialog. Locally, set `DATASET_ZIP_PATH` to the ZIP path before running.


In [ ]:
# Local usage: set this manually, for example:
# DATASET_ZIP_PATH = Path("/path/to/chicken_threat_dataset_v3.zip")
DATASET_ZIP_PATH = None


def clean_dir(path: Path) -> None:
    path = path.resolve()
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def upload_zip_in_colab() -> Path:
    if not IN_COLAB:
        raise RuntimeError("Set DATASET_ZIP_PATH when running locally.")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one dataset zip.")
    uploaded_name = next(iter(uploaded))
    return Path(uploaded_name).resolve()


def find_dataset_root(root: Path) -> Path:
    candidates = [root, *[path for path in root.rglob("data.yaml")]]
    for candidate in candidates:
        dataset_root = candidate.parent if candidate.name == "data.yaml" else candidate
        if (dataset_root / "data.yaml").exists():
            if all((dataset_root / split / "images").exists() for split in ("train", "valid")):
                return dataset_root.resolve()
    raise FileNotFoundError(f"Could not find a YOLO data.yaml with train/valid folders under {root}")


def rewrite_data_yaml(dataset_root: Path) -> Path:
    data_yaml = dataset_root / "data.yaml"
    data = yaml.safe_load(data_yaml.read_text(encoding="utf-8")) or {}
    data["path"] = str(dataset_root.resolve())
    data.setdefault("train", "train/images")
    data.setdefault("val", "valid/images")
    data.setdefault("test", "test/images")
    data["nc"] = len(data.get("names", CANONICAL_NAMES))
    data_yaml.write_text(yaml.safe_dump(data, sort_keys=False), encoding="utf-8")
    return data_yaml


zip_path = Path(DATASET_ZIP_PATH).resolve() if DATASET_ZIP_PATH else upload_zip_in_colab()
if not zip_path.exists() or zip_path.suffix.lower() != ".zip":
    raise FileNotFoundError(f"Dataset ZIP not found: {zip_path}")

clean_dir(IMPORT_ROOT)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(IMPORT_ROOT)

DATASET_ROOT = find_dataset_root(IMPORT_ROOT)
DATA_YAML = rewrite_data_yaml(DATASET_ROOT)
print(f"Dataset root: {DATASET_ROOT}")
print(f"data.yaml: {DATA_YAML}")


## Dataset validation and class balance

In [ ]:
def load_class_names(dataset_root: Path) -> list[str]:
    data = yaml.safe_load((dataset_root / "data.yaml").read_text(encoding="utf-8"))
    names = data.get("names")
    if isinstance(names, dict):
        return [str(names[key]) for key in sorted(names, key=lambda key: int(key))]
    if isinstance(names, list):
        return [str(name) for name in names]
    raise ValueError("data.yaml has unsupported names format")


def image_files(image_dir: Path) -> list[Path]:
    if not image_dir.exists():
        return []
    return sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)


def clip_yolo_box(box: Iterable[float]) -> tuple[float, float, float, float] | None:
    x_center, y_center, width, height = [float(value) for value in box]
    if width <= 0 or height <= 0:
        return None
    x_min = max(0.0, x_center - width / 2)
    y_min = max(0.0, y_center - height / 2)
    x_max = min(1.0, x_center + width / 2)
    y_max = min(1.0, y_center + height / 2)
    width = x_max - x_min
    height = y_max - y_min
    if width <= 0 or height <= 0:
        return None
    return ((x_min + x_max) / 2, (y_min + y_max) / 2, width, height)


def parse_yolo_line(line: str) -> tuple[int, tuple[float, float, float, float]] | None:
    parts = line.split()
    if len(parts) != 5:
        return None
    try:
        class_id = int(float(parts[0]))
        coords = [float(value) for value in parts[1:]]
    except ValueError:
        return None
    box = clip_yolo_box(coords)
    if box is None:
        return None
    return class_id, box


def summarize_dataset(dataset_root: Path) -> pd.DataFrame:
    names = load_class_names(dataset_root)
    rows = []
    for split in SPLITS:
        image_dir = dataset_root / split / "images"
        label_dir = dataset_root / split / "labels"
        counts = Counter()
        invalid = 0
        missing = 0
        empty = 0
        for image_path in image_files(image_dir):
            label_path = label_dir / f"{image_path.stem}.txt"
            if not label_path.exists():
                missing += 1
                continue
            text = label_path.read_text(encoding="utf-8").strip()
            if not text:
                empty += 1
                continue
            for line in text.splitlines():
                parsed = parse_yolo_line(line)
                if parsed is None:
                    invalid += 1
                    continue
                class_id, _ = parsed
                label = names[class_id] if 0 <= class_id < len(names) else f"unknown:{class_id}"
                counts[label] += 1
        for name in names:
            rows.append({"split": split, "class": name, "objects": counts[name]})
        rows.append({"split": split, "class": "__images__", "objects": len(image_files(image_dir))})
        rows.append({"split": split, "class": "__missing_label_files__", "objects": missing})
        rows.append({"split": split, "class": "__empty_label_files__", "objects": empty})
        rows.append({"split": split, "class": "__invalid_labels__", "objects": invalid})
    return pd.DataFrame(rows)


def validate_dataset(dataset_root: Path) -> list[str]:
    errors = []
    names = load_class_names(dataset_root)
    if names != CANONICAL_NAMES:
        errors.append(f"Class names differ from expected canonical names: {names}")
    for split in SPLITS:
        image_dir = dataset_root / split / "images"
        label_dir = dataset_root / split / "labels"
        if not image_dir.exists():
            errors.append(f"Missing image directory: {image_dir}")
        if not label_dir.exists():
            errors.append(f"Missing label directory: {label_dir}")
        image_stems = {path.stem for path in image_files(image_dir)}
        label_stems = {path.stem for path in label_dir.glob("*.txt")} if label_dir.exists() else set()
        if image_stems - label_stems:
            errors.append(f"{split}: {len(image_stems - label_stems)} missing label files")
        if label_stems - image_stems:
            errors.append(f"{split}: {len(label_stems - image_stems)} orphan label files")
        for label_path in sorted(label_dir.glob("*.txt")) if label_dir.exists() else []:
            for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
                if not line.strip():
                    continue
                parsed = parse_yolo_line(line)
                if parsed is None:
                    errors.append(f"{label_path}:{line_number}: invalid label row")
                    continue
                class_id, box = parsed
                if not 0 <= class_id < len(names):
                    errors.append(f"{label_path}:{line_number}: class id out of range")
                if any(value < 0 or value > 1 for value in box):
                    errors.append(f"{label_path}:{line_number}: box outside 0..1")
    return errors


summary_df = summarize_dataset(DATASET_ROOT)
display(summary_df.pivot(index="class", columns="split", values="objects").fillna(0).astype(int))

validation_errors = validate_dataset(DATASET_ROOT)
if validation_errors:
    print("Validation failed:")
    for error in validation_errors[:100]:
        print(" -", error)
    raise RuntimeError(f"Dataset validation failed with {len(validation_errors)} errors.")
print("Validation passed.")


In [ ]:
plot_df = summary_df[~summary_df["class"].str.startswith("__")]
pivot = plot_df.pivot(index="class", columns="split", values="objects").fillna(0)
pivot.loc[CANONICAL_NAMES].plot(kind="bar", stacked=True, figsize=(14, 5))
plt.title("v3 dataset object balance")
plt.ylabel("objects")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Training configuration

The defaults are aimed at improving generalization without making Colab training unnecessarily fragile:

- More epochs with early stopping.
- Auto batch sizing.
- Cosine learning-rate schedule.
- Mixed precision when CUDA supports it.
- HSV, affine, mosaic, mixup, and light copy-paste augmentation.
- `close_mosaic` near the end, so final epochs see more natural images.

For quick smoke tests, set `EPOCHS = 3` and `MODEL_WEIGHTS = "yolo11n.pt"`.


In [ ]:
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_WEIGHTS = "yolo11s.pt"   # switch to yolo11n.pt for faster/cheaper experiments
RUN_NAME = "v3_yolo11s_aug_balanced"
EPOCHS = 150
IMG_SIZE = 640
PATIENCE = 30

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

TRAIN_ARGS = dict(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=-1,
    patience=PATIENCE,
    project=str(RUNS_ROOT),
    name=RUN_NAME,
    pretrained=True,
    seed=SEED,
    deterministic=False,
    workers=2,
    cache="disk",
    device=0 if torch.cuda.is_available() else "cpu",
    amp=True,
    cos_lr=True,
    optimizer="auto",
    lr0=0.003,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    close_mosaic=15,
    # Built-in YOLO augmentations.
    hsv_h=0.015,
    hsv_s=0.65,
    hsv_v=0.35,
    degrees=7.0,
    translate=0.08,
    scale=0.45,
    shear=2.0,
    perspective=0.0005,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.75,
    mixup=0.10,
    copy_paste=0.05,
    # Detection loss balance. Defaults are good, but keeping them explicit documents the recipe.
    box=7.5,
    cls=0.5,
    dfl=1.5,
)
TRAIN_ARGS


## Train model

In [ ]:
model = YOLO(MODEL_WEIGHTS)
results = model.train(**TRAIN_ARGS)

RUN_DIR = Path(results.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
print(f"Run directory: {RUN_DIR}")
print(f"Best weights: {BEST_WEIGHTS}")
print(f"Last weights: {LAST_WEIGHTS}")


## Evaluate validation and test performance

In [ ]:
def evaluate_split(weights_path: Path, split: str):
    eval_model = YOLO(str(weights_path))
    metrics = eval_model.val(
        data=str(DATA_YAML),
        split=split,
        imgsz=IMG_SIZE,
        project=str(RUNS_ROOT),
        name=f"{RUN_NAME}_eval_{split}",
        plots=True,
        conf=0.001,
        iou=0.6,
    )
    print(f"\n{split.upper()} metrics")
    print(f"mAP50-95: {metrics.box.map:.4f}")
    print(f"mAP50:    {metrics.box.map50:.4f}")
    print(f"mAP75:    {metrics.box.map75:.4f}")
    return metrics

if not BEST_WEIGHTS.exists() and LAST_WEIGHTS.exists():
    BEST_WEIGHTS = LAST_WEIGHTS
if not BEST_WEIGHTS.exists():
    raise FileNotFoundError("No trained weights found. Run training first.")

valid_metrics = evaluate_split(BEST_WEIGHTS, "val")
test_metrics = evaluate_split(BEST_WEIGHTS, "test")


In [ ]:
def per_class_map_table(metrics, split_name: str) -> pd.DataFrame:
    names = metrics.names if hasattr(metrics, "names") else {i: name for i, name in enumerate(CANONICAL_NAMES)}
    maps = getattr(metrics.box, "maps", [])
    rows = []
    for i, value in enumerate(maps):
        rows.append({"split": split_name, "class": names.get(i, str(i)) if isinstance(names, dict) else names[i], "mAP50-95": float(value)})
    return pd.DataFrame(rows)

per_class_df = pd.concat([
    per_class_map_table(valid_metrics, "valid"),
    per_class_map_table(test_metrics, "test"),
], ignore_index=True)
display(per_class_df.pivot(index="class", columns="split", values="mAP50-95").loc[CANONICAL_NAMES])


## Inspect evaluation plots

In [ ]:
def show_image_if_exists(path: Path, title: str) -> None:
    if path.exists():
        image = Image.open(path)
        plt.figure(figsize=(9, 7))
        plt.imshow(image)
        plt.axis("off")
        plt.title(title)
        plt.show()
    else:
        print(f"Not found: {path}")

show_image_if_exists(RUNS_ROOT / f"{RUN_NAME}_eval_test" / "confusion_matrix.png", "Test confusion matrix")
show_image_if_exists(RUNS_ROOT / f"{RUN_NAME}_eval_test" / "PR_curve.png", "Test PR curve")


## Export best model package

In [ ]:
MODEL_EXPORT_DIR = EXPORT_ROOT / RUN_NAME
MODEL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for source in [BEST_WEIGHTS, LAST_WEIGHTS, DATA_YAML, RUN_DIR / "args.yaml", RUN_DIR / "results.csv"]:
    if Path(source).exists():
        shutil.copy2(source, MODEL_EXPORT_DIR / Path(source).name)

# Also save a compact metrics summary.
metrics_summary = {
    "model_weights": MODEL_WEIGHTS,
    "run_name": RUN_NAME,
    "best_weights": str(BEST_WEIGHTS),
    "valid_map50_95": float(valid_metrics.box.map),
    "valid_map50": float(valid_metrics.box.map50),
    "test_map50_95": float(test_metrics.box.map),
    "test_map50": float(test_metrics.box.map50),
    "class_names": CANONICAL_NAMES,
}
(MODEL_EXPORT_DIR / "metrics_summary_v3.yaml").write_text(yaml.safe_dump(metrics_summary, sort_keys=False), encoding="utf-8")

zip_file = Path(shutil.make_archive(str(EXPORT_ROOT / RUN_NAME), "zip", root_dir=MODEL_EXPORT_DIR))
print(f"Model package: {zip_file}")
print(f"Size: {zip_file.stat().st_size / (1024 * 1024):.1f} MB")

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_file))
else:
    print(zip_file)


## Inference sanity check on random test images

In [ ]:
sample_dir = DATASET_ROOT / "test" / "images"
sample_images = image_files(sample_dir)
if not sample_images:
    print("No test images found.")
else:
    selected = random.sample(sample_images, k=min(6, len(sample_images)))
    inference_model = YOLO(str(BEST_WEIGHTS))
    predictions = inference_model.predict(source=[str(path) for path in selected], conf=0.25, imgsz=IMG_SIZE)

    columns = 3
    rows = (len(predictions) + columns - 1) // columns
    plt.figure(figsize=(5 * columns, 4 * rows))
    for index, result in enumerate(predictions, start=1):
        rendered = result.plot()[..., ::-1]
        plt.subplot(rows, columns, index)
        plt.imshow(rendered)
        plt.axis("off")
        plt.title(Path(result.path).name[:35])
    plt.tight_layout()
    plt.show()
